# Generate Workout

## Import Libraries

In [18]:
import numpy as np
import pandas as pd

## Load JSON Data

In [19]:
exercises_data = pd.read_json('../../data/exercises.json')

## Feature Engineering

In [20]:
# ==========================================
# FEATURE ENGINEERING: ENVIRONMENT CLASSIFICATION
# ==========================================
# Objective: To bridge the gap between raw exercise data and user constraints.
# We must programmatically determine which exercises are feasible for a 
# "Home" setting vs. those requiring a "Gym".

# 1. Sample JSON Data Structure (assumed input)
# ... (Data loading logic) ...

# 2. Create Exercise DataFrame
df_exercises_complete = pd.DataFrame(exercises_data)

# --- CLASSIFICATION LOGIC ---

def categorize_environment(equipments_list):
    """
    Classifies the required environment ('Home' or 'Gym') based on equipment type.

    This function applies a heuristic rule to sort exercises. It assumes that 
    bodyweight movements and portable resistance bands are suitable for home use, 
    while free weights (dumbbells/barbells) and machines require a gym setting.

    ---------------------------------------------------------------------------
    Classification Logic:
    
    1. Home Environment:
       - Triggers: 'Body Weight' (No equipment) OR 'Band' (Portable resistance).
       - Reasoning: minimal space and investment required.

    2. Gym Environment:
       - Triggers: Everything else (Default).
       - Includes: Dumbbells, Barbells, Cables, Machines, Kettlebells.
       - Reasoning: These items typically require a dedicated facility or 
         significant home equipment investment.
    
    ---------------------------------------------------------------------------
    Args:
        equipments_list (list): A list of equipment strings (e.g., ['Dumbbell', 'Bench']).
        *Note: The function analyzes the primary equipment (index 0).*

    Returns:
        str: 'Home' or 'Gym'.
    """

    # Analyze the Primary Equipment (First item in the list)
    eq = equipments_list[0].lower()
    
    # Check for Home-friendly keywords
    if 'body weight' in eq or 'band' in eq:
        return 'Home'
    else:
        # Default to Gym for all external weights/machines
        return 'Gym'

# --- APPLY LOGIC ---
# Create a new column 'Environment' to drive the recommendation engine
df_exercises_complete['Environment'] = df_exercises_complete['equipments'].apply(categorize_environment)

print("Dataset Ready for Processing.")

Dataset Ready for Processing.


## Data Cleaning

In [21]:
# ==========================================
# DATA CLEANING: TYPE ENFORCEMENT
# ==========================================
# Objective: To ensure data consistency before processing.
# 
# Why is this necessary?
# Raw datasets often contain null values (NaN) or inconsistent types (e.g., a number 
# inside a text column). If we try to run string operations (like .lower()) on a 
# NaN value, the program will crash. These functions act as a "Safety Guard".

# A. List-Type Columns (Requiring Iteration)
list_columns = ['targetMuscles', 'bodyParts', 'equipments', 'secondaryMuscles', 'instructions']

# B. String-Type Columns (Requiring Text Manipulation)
text_columns = ['gifUrl', 'name', 'exerciseId']

# --- HELPER FUNCTIONS ---

def fix_list_column(val):
    """
    Enforces a List data type for iterable columns.

    This function handles missing values (NaN) or malformed data by converting them 
    into a safe default list. This prevents 'TypeError' crashes when the algorithm 
    attempts to iterate through tags or equipment lists later.

    ---------------------------------------------------------------------------
    Args:
        val (any): The raw cell value (could be list, float/NaN, or string).

    Returns:
        list: A valid list. Returns ['none'] if the input was null or invalid.
    """
    if isinstance(val, list):
        return val
    # Fallback for NaN or non-list types
    return ['none'] 

def fix_text_column(val):
    """
    Enforces a String data type for text-based columns.

    This function ensures that missing values (NaN), which Pandas often treats as 
    floats, are converted to valid strings. This allows subsequent text operations 
    (like .lower(), .strip(), or regex) to run without errors.

    ---------------------------------------------------------------------------
    Args:
        val (any): The raw cell value.

    Returns:
        str: A valid string. Returns "None" if the input was null (NaN).
    """
    if pd.isna(val):
        return "None" 
    return str(val)

# --- BATCH APPLICATION (CLEANING PIPELINE) ---

print("Cleaning Dataset...")

# 1. Apply List Fixer
for col in list_columns:
    # Check if column exists to prevent KeyErrors
    if col in df_exercises_complete.columns:
        df_exercises_complete[col] = df_exercises_complete[col].apply(fix_list_column)

# 2. Apply Text Fixer
for col in text_columns:
    if col in df_exercises_complete.columns:
        df_exercises_complete[col] = df_exercises_complete[col].apply(fix_text_column)

print("Data Type Enforcement Complete.")

Cleaning Dataset...
Data Type Enforcement Complete.


## Saving Dataset

In [22]:
df_exercises_complete.to_csv('../../data/dataset_workout.csv', index=False)